In [1]:
#!/usr/bin/env python
from uproot_methods import TVector2Array, TLorentzVectorArray
from optparse import OptionParser
from coffea.jetmet_tools import FactorizedJetCorrector, JetCorrectionUncertainty, JetTransformer, JetResolution, JetResolutionScaleFactor
from coffea.util import load, save
from coffea import hist, processor
from coffea.arrays import Initialize
import lz4.frame as lz4f
import cloudpickle
import json
import pprint
import numpy as np
import math
import awkward
np.seterr(divide='ignore', invalid='ignore', over='ignore')
import time

In [2]:
import awkward as ak
from coffea.nanoaod.nanoevents import NanoEvents

# fnameUL = "root://xrootd-cms.infn.it//store/mc/RunIISummer20UL18NanoAODv9/DYJetsToLL_M-50_HT-400to600_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/130000/4F3BA45A-907E-9846-AECE-578D2AF50F44.root"
# fnameL = 'root://cmseos.fnal.gov//store/group/lpccoffea/coffeabeans/NanoAODv6/nano_2018/TTJets_TuneCP5_13TeV-amcatnloFXFX-pythia8/NanoTuples-2018_RunIIAutumn18MiniAOD-102X_v15-v2-ext1/191208_161323/0002/nano_2015.root'

# eventsL = NanoEvents.from_file(fnameL)

fnameUL =             "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_Pt-100To250_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/100000/8A16FAF9-40C4-9042-A888-CC6FF79132BD.root"
events = NanoEvents.from_file(fnameUL)

# Jet PUID

In [3]:

j = events.Jet
# j['isgood'] = isGoodJet(j.pt, j.eta, j.jetId, j.puId, j.neHEF, j.chHEF)
# j['isHEM'] = isHEMJet(j.pt, j.eta, j.phi)
# j['isclean'] = ~match(j, e_loose, 0.4) & ~match( j, mu_loose, 0.4) & ~match(j, pho_loose, 0.4)

In [9]:
((j.btagDeepFlavB>=0.2783).sum()).sum()

21876

In [12]:
j.jetId

<JaggedArray [[6 6 6 6] [6 6 6] [6 6 6 0 6] ... [6 6] [6 6 6 6] [2 6 6 ... 6 6 6]] at 0x7f528752f5c0>

<JaggedArray [[0 0 0 0] [0 0 0] [0 0 0 0 0] ... [0 0] [0 0 0 0] [0 0 0 ... 0 0 0]] at 0x7f5287552c88>

In [63]:
j.jetId&2

<JaggedArray [[2 2 2 2] [2 2 2] [2 2 2 0 2] ... [2 2] [2 2 2 2] [2 2 2 ... 2 2 2]] at 0x7f52866d4e10>

In [64]:
arr = np.array([6,6,6])
arr

array([6, 6, 6])

In [65]:
arr

array([6, 6, 6])

In [67]:
arr&2

array([2, 2, 2])

In [41]:
len((j.puId>0).flatten())

710218

In [26]:
j.pt<50

<JaggedArray [[False True True True] [False True True] [False True True True True] ... [False False] [False False True True] [False False False ... True True True]] at 0x7f52866d4240>

In [49]:
(j.puId>0) & (j.pt<50)

<JaggedArray [[False True True True] [False True True] [False True True True True] ... [False False] [False False True True] [False False False ... True True True]] at 0x7f52866d4ef0>

In [75]:
mask = (j.pt>=30) & ((j.jetId&2)==2)
mask = ((j.pt>=50)&mask) | ((j.pt<50)&mask&((j.puId>0)))

In [25]:
j_ntot = j.counts
j_ntot>=1

array([ True,  True,  True, ...,  True,  True,  True])

In [26]:
(j.pt>=50).any()

array([ True,  True,  True, ...,  True,  True,  True])

In [78]:
j

<JetArray [[<Jet 0> <Jet 1> <Jet 2> <Jet 3>] [<Jet 4> <Jet 5> <Jet 6>] [<Jet 7> <Jet 8> <Jet 9> <Jet 10> <Jet 11>] ... [<Jet 710204> <Jet 710205>] [<Jet 710206> <Jet 710207> <Jet 710208> <Jet 710209>] [<Jet 710210> <Jet 710211> <Jet 710212> ... <Jet 710215> <Jet 710216> <Jet 710217>]] at 0x7f52873fada0>

# dphi(met, leading jet)

In [3]:
met = events.MET
met['T'] = TVector2Array.from_polar(met.pt, met.phi)
j = j = events.Jet

j['T'] = TVector2Array.from_polar(j.pt, j.phi)
j['p4'] = TLorentzVectorArray.from_ptetaphim(j.pt, j.eta, j.phi, j.mass)
leading_j = j[j.pt.argmax()]


In [32]:
Delta_Phi_Met_LJ = (abs(met.T.delta_phi(leading_j.T)).flatten())>1.5

In [33]:
Delta_Phi_Met_LJ

array([ True,  True,  True, ...,  True,  True, False])

In [35]:
(abs(met.T.delta_phi(leading_j.T)).sum())>1.5

array([ True,  True,  True, ...,  True,  True, False])

In [23]:
a = (abs(leading_j.T.delta_phi(met.T)).min()).flatten()

array([ True,  True,  True, ...,  True,  True, False])

In [25]:
b = (abs(leading_j.T.delta_phi(met.T)).sum()).flatten()

In [29]:
(a==b).sum()

133026

In [22]:
abs(met.T.delta_phi(leading_j.T)).max().sum()

323298.38

In [23]:
abs(met.T.delta_phi(leading_j.T)).sum().sum()

323298.38

In [24]:
ids         = load('../data/ids.coffea')


In [31]:
ids

{'isLooseElectron': <function __main__.isLooseElectron(pt, eta, dxy, dz, veto_id, year)>,
 'isTightElectron': <function __main__.isTightElectron(pt, eta, dxy, dz, tight_id, year)>,
 'isLooseMuon': <function __main__.isLooseMuon(pt, eta, iso, loose_id, year)>,
 'isTightMuon': <function __main__.isTightMuon(pt, eta, iso, tight_id, year)>,
 'isLooseTau': <function __main__.isLooseTau(pt, eta, decayMode, _id, year)>,
 'isLoosePhoton': <function __main__.isLoosePhoton(pt, eta, loose_id, year)>,
 'isTightPhoton': <function __main__.isTightPhoton(pt, tight_id, year)>,
 'isGoodJet': <function __main__.isGoodJet(pt, eta, jet_id, pu_id, nhf, chf)>,
 'isGoodFatJet': <function __main__.isGoodFatJet(pt, eta, jet_id)>,
 'isHEMJet': <function __main__.isHEMJet(pt, eta, phi)>}

# testing the .sum(), .min(), .max() methods

In [3]:
corrections = load('../data/corrections.coffea')
ids         = load('../data/ids.coffea')
common      = load('../data/common.coffea')



<function __main__.match(a, b, val)>

In [4]:
match = common['match']
isGoodJet = ids['isGoodJet']
isLooseElectron = ids['isLooseElectron']
isTightElectron = ids['isTightElectron']
isLooseMuon = ids['isLooseMuon']
isTightMuon = ids['isTightMuon']
isLooseTau = ids['isLooseTau']
isLoosePhoton = ids['isLoosePhoton']
isTightPhoton = ids['isTightPhoton']
isGoodJet = ids['isGoodJet']
isHEMJet = ids['isHEMJet']

In [5]:
met = events.MET
# if '2018' == '2017':
#     events.METFixEE2017  # Recommended for 2017
met['T'] = TVector2Array.from_polar(met.pt, met.phi)
calomet = events.CaloMET
puppimet = events.PuppiMET

###
# Initialize physics objects
###

mu = events.Muon
mu['isloose'] = isLooseMuon(mu.pt, mu.eta, mu.pfRelIso04_all, mu.looseId, '2018')
mu['istight'] = isTightMuon(mu.pt, mu.eta, mu.pfRelIso04_all, mu.tightId, '2018')
mu['T'] = TVector2Array.from_polar(mu.pt, mu.phi)
mu['p4'] = TLorentzVectorArray.from_ptetaphim(mu.pt, mu.eta, mu.phi, mu.mass)
mu_loose = mu[mu.isloose.astype(np.bool)]
mu_tight = mu[mu.istight.astype(np.bool)]
mu_ntot = mu.counts
mu_nloose = mu_loose.counts
mu_ntight = mu_tight.counts
leading_mu = mu[mu.pt.argmax()]
leading_mu = leading_mu[leading_mu.istight.astype(np.bool)]

e = events.Electron
event_size = len(e)
e['isclean'] = ~match(e, mu_loose, 0.3)
e['isloose'] = isLooseElectron(e.pt, e.eta+e.deltaEtaSC, e.dxy, e.dz, e.cutBased, '2018')
e['istight'] = isTightElectron(e.pt, e.eta+e.deltaEtaSC, e.dxy, e.dz, e.cutBased, '2018')
e['T'] = TVector2Array.from_polar(e.pt, e.phi)
e['p4'] = TLorentzVectorArray.from_ptetaphim(e.pt, e.eta, e.phi, e.mass)
e_clean = e[e.isclean.astype(np.bool)]
e_loose = e_clean[e_clean.isloose.astype(np.bool)]
e_tight = e_clean[e_clean.istight.astype(np.bool)]
e_ntot = e.counts
e_nloose = e_loose.counts
e_ntight = e_tight.counts
leading_e = e[e.pt.argmax()]
leading_e = leading_e[leading_e.isclean.astype(np.bool)]
leading_e = leading_e[leading_e.istight.astype(np.bool)]


tau = events.Tau
tau['isclean'] = ~match(tau, mu_loose, 0.4) & ~match(tau, e_loose, 0.4)
#         tau['isloose'] = isLooseTau(tau.pt, tau.eta, tau.idDecayMode, tau.idMVAoldDM2017v2, '2018')
try:
    tau['isloose']=isLooseTau(tau.pt,tau.eta,tau.idDecayMode,tau.idDeepTau2017v2p1VSjet,'2018')
except:
    tau['isloose']=isLooseTau(tau.pt,tau.eta,tau.idDecayModeOldDMs,tau.idDeepTau2017v2p1VSjet,'2018')
else: 
    tau['isloose']=isLooseTau(tau.pt,tau.eta,tau.idDecayModeNewDMs,tau.idDeepTau2017v2p1VSjet,'2018')

tau_clean = tau[tau.isclean.astype(np.bool)]
tau_loose = tau_clean[tau_clean.isloose.astype(np.bool)]
tau_ntot = tau.counts
tau_nloose = tau_loose.counts

pho = events.Photon
pho['isclean'] = ~match(pho, mu_loose, 0.5) & ~match(pho, e_loose, 0.5)
_id = 'cutBased'
# if '2018' == '2016':
#     _id = 'cutBased'
pho['isloose'] = isLoosePhoton(pho.pt, pho.eta, pho[_id], '2018') & (pho.electronVeto)  # added electron veto flag
pho['istight'] = isTightPhoton(pho.pt, pho[_id], '2018') & (pho.isScEtaEB) & (pho.electronVeto)  # tight photons are barrel only
pho['T'] = TVector2Array.from_polar(pho.pt, pho.phi)
pho_clean = pho[pho.isclean.astype(np.bool)]
pho_loose = pho_clean[pho_clean.isloose.astype(np.bool)]
pho_tight = pho_clean[pho_clean.istight.astype(np.bool)]
pho_ntot = pho.counts
pho_nloose = pho_loose.counts
pho_ntight = pho_tight.counts
leading_pho = pho[pho.pt.argmax()]
leading_pho = leading_pho[leading_pho.isclean.astype(np.bool)]
leading_pho = leading_pho[leading_pho.istight.astype(np.bool)]


In [6]:



j = events.Jet
j['isgood'] = isGoodJet(j.pt, j.eta, j.jetId, j.puId, j.neHEF, j.chHEF)
# j['isHEM'] = isHEMJet(j.pt, j.eta, j.phi)
j['isclean'] = ~match(j,e_loose,0.4)&~match(j,mu_loose,0.4)&~match(j,pho_loose,0.4)
#j['isiso'] = ~match(j,fj_clean[fj_clean.pt.argmax()],1.5)
# j['isdcsvL'] = (j.btagDeepB>deepcsvWPs['loose'])
# j['isdflvL'] = (j.btagDeepFlavB>deepflavWPs['loose'])
# j['isdflvM'] = (j.btagDeepFlavB > deepflavWPs['medium']) ## from Rishab
# j['isdcsvM'] = (j.btagDeepB > deepcsvWPs['medium']) ## from Rishabh
j['T'] = TVector2Array.from_polar(j.pt, j.phi)
j['p4'] = TLorentzVectorArray.from_ptetaphim(j.pt, j.eta, j.phi, j.mass)
j['ptRaw'] =j.pt * (1-j.rawFactor)
j['massRaw'] = j.mass * (1-j.rawFactor)
j['rho'] = j.pt.ones_like()*events.fixedGridRhoFastjetAll.array
j_good = j[j.isgood.astype(np.bool)]
j_clean = j_good[j_good.isclean.astype(np.bool)]
# j_iso = j_clean[j_clean.isiso.astype(np.bool)]
#j_iso = j_clean[j_clean.astype(np.bool)]    
j_iso = j_good[j_good.isclean.astype(np.bool)]   
# j_dcsvL = j_clean[j_clean.isdcsvL.astype(np.bool)]
# j_dflvL = j_clean[j_clean.isdflvL.astype(np.bool)]
# j_dflvM = j_clean[j_clean.isdflvM.astype(np.bool)] 
# j_dcsvM = j_clean[j_clean.isdcsvM.astype(np.bool)] 
# j_HEM = j[j.isHEM.astype(np.bool)]
j_ntot=j.counts
j_ngood=j_good.counts
j_nclean=j_clean.counts
#        j_niso=j_iso.counts
# j_ndcsvL=j_dcsvL.counts
# j_ndflvL=j_dflvL.counts
# j_ndflvM = j_dflvM.counts 
# j_ndcsvM = j_dcsvM.counts 
# j_nHEM = j_HEM.counts
leading_j = j[j.pt.argmax()]
leading_j = leading_j[leading_j.isgood.astype(np.bool)]
leading_j = leading_j[leading_j.isclean.astype(np.bool)]

In [37]:
Delta_Phi_Met_LJ_sum = (abs(leading_j.T.delta_phi(met.T)).sum())
Delta_Phi_Met_LJ_max = (abs(leading_j.T.delta_phi(met.T)).max())
Delta_Phi_Met_LJ_min = (abs(leading_j.T.delta_phi(met.T)).min())
Delta_Phi_Met_LJ_flatten = (abs(leading_j.T.delta_phi(met.T)).flatten())

In [38]:
print('Delta_Phi_Met_LJ_sum:', (Delta_Phi_Met_LJ_sum).sum())
print('Delta_Phi_Met_LJ_min:', (Delta_Phi_Met_LJ_min).sum())
print('Delta_Phi_Met_LJ_max:', (Delta_Phi_Met_LJ_max).sum())
print('Delta_Phi_Met_LJ_flatten:', Delta_Phi_Met_LJ_flatten.sum())


Delta_Phi_Met_LJ_sum: 245915.9
Delta_Phi_Met_LJ_min: inf
Delta_Phi_Met_LJ_max: -inf
Delta_Phi_Met_LJ_flatten: 245915.89


In [41]:
print(abs(leading_j.T.delta_phi(met.T)), len(abs(leading_j.T.delta_phi(met.T))))
print(Delta_Phi_Met_LJ_sum)
print(Delta_Phi_Met_LJ_min)
print(Delta_Phi_Met_LJ_max)
print(Delta_Phi_Met_LJ_flatten, len(Delta_Phi_Met_LJ_flatten))

[[3.0496647] [2.8039372] [2.7904665] ... [2.7857487] [2.2124176] []] 133026
[3.0496647 2.8039372 2.7904665 ... 2.7857487 2.2124176 0.       ]
[3.0496647 2.8039372 2.7904665 ... 2.7857487 2.2124176       inf]
[3.0496647 2.8039372 2.7904665 ... 2.7857487 2.2124176      -inf]
[3.0496647 2.8039372 2.7904665 ... 2.904279  2.7857487 2.2124176] 93067


In [48]:
met.T.delta_phi(leading_j.T.sum())

array([ 3.0496647, -2.8039372, -2.7904663, ...,  2.7857487,  2.2124178,
        3.1323245], dtype=float32)

In [12]:
abs(leading_j.T.delta_phi(met.T))

<JaggedArray [[3.0496647] [2.8039372] [2.7904665] ... [2.7857487] [2.2124176] []] at 0x7fc022b1c7f0>

In [13]:
leading_j.pt

<JaggedArray [[110.0] [63.21875] [105.3125] ... [210.75] [82.375] []] at 0x7fc022b20898>

In [77]:
j.pt

<JaggedArray [[110.0 40.40625 20.46875 15.3203125] [63.21875 28.890625 16.328125] [105.3125 47.03125 22.421875 18.171875 17.5625] ... [210.75 59.5625] [82.375 76.9375 15.6796875 15.1640625] [146.75 101.0625 53.75 ... 22.125 21.8125 16.53125]] at 0x7f3dcc8f17f0>

In [14]:
j_good.pt

<JaggedArray [[110.0 40.40625] [63.21875] [105.3125] ... [210.75 59.5625] [82.375] [101.0625 53.25 36.84375]] at 0x7fc022b202e8>

In [15]:
j_clean.pt

<JaggedArray [[110.0 40.40625] [63.21875] [105.3125] ... [210.75 59.5625] [82.375] [101.0625 53.25 36.84375]] at 0x7fc006c9d5c0>

In [17]:
j_good[j_good.isclean.astype(np.bool)].pt


<JaggedArray [[110.0 40.40625] [63.21875] [105.3125] ... [210.75 59.5625] [82.375] [101.0625 53.25 36.84375]] at 0x7fc06f9d3da0>

In [20]:
j_good.isclean.astype(np.bool)

<JaggedArray [[True True] [True] [True] ... [True True] [True] [True True True]] at 0x7fc006a9c9e8>

In [24]:
(j_clean[j_clean.pt.argmax()].pt)

<JaggedArray [[110.0] [63.21875] [105.3125] ... [210.75] [82.375] [101.0625]] at 0x7fc006a9c0f0>

In [26]:
leading_j.pt

<JaggedArray [[110.0] [63.21875] [105.3125] ... [210.75] [82.375] []] at 0x7fc0680b8518>

In [28]:
(j[j.pt.argmax()]).pt

<JaggedArray [[110.0] [63.21875] [105.3125] ... [210.75] [82.375] [146.75]] at 0x7fc006a9cd30>

In [29]:
(leading_j[leading_j.isgood.astype(np.bool)]).pt

<JaggedArray [[110.0] [63.21875] [105.3125] ... [210.75] [82.375] []] at 0x7fc022cbeba8>

In [30]:
(leading_j[leading_j.isclean.astype(np.bool)]).pt

<JaggedArray [[110.0] [63.21875] [105.3125] ... [210.75] [82.375] []] at 0x7fc006b10fd0>

In [56]:
(leading_e.pt.sum()).sum()

1897751.6

In [51]:
leading_e_clean = e_tight[e_tight.pt.argmax()]


In [55]:
(leading_e_clean.pt.sum()).sum()

1935527.6